# GraphForge — Rejection-Sampling SFT on a free Colab T4

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nithin062006/scaler/blob/main/training/notebook.ipynb)

This notebook runs the full pipeline end-to-end against the GraphForge OpenEnv environment:

1. Clone the repo and install deps (1 min on T4)
2. Baseline-eval Qwen2.5-0.5B-Instruct against the tier-0 task
3. Generate trajectories (oracle + live model) and reject-sample
4. SFT the kept trajectories (TRL SFTTrainer + LoRA)
5. Trained-eval the same model and write all hackathon plots

Expected wall-clock: ~10–20 min on a T4.

## 1. Setup

In [ ]:
import os, subprocess, pathlib

REPO_URL = 'https://github.com/nithin062006/scaler.git'
cwd = pathlib.Path(os.getcwd())

# Idempotent: handles fresh runtime, restarted runtime, and re-runs.
if (cwd / 'graphforge').exists() and (cwd / 'env').exists():
    print(f'Already inside repo: {cwd}')
elif (cwd / 'graphforge_repo').exists():
    os.chdir('graphforge_repo')
    print(f'Cd-ed into existing clone: {os.getcwd()}')
else:
    subprocess.check_call(['git', 'clone', '-q', REPO_URL, 'graphforge_repo'])
    os.chdir('graphforge_repo')
    print(f'Cloned + cd-ed: {os.getcwd()}')

print(os.listdir('.'))

In [ ]:
# Install runtime + training deps. peft & trl handle the SFT side.
%pip install -q -e ".[training]"
%pip install -q peft
import torch
print('CUDA available:', torch.cuda.is_available())
print('Device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

In [ ]:
# Patch HfPolicy.sample for newer transformers' apply_chat_template return type.
# Idempotent — overwrites the file on every run. Safe to keep even after the
# fix lands upstream (writing the same content is a no-op observationally).
import pathlib, sys

_FIXED = '''"""Policy interface and stub policies."""

from __future__ import annotations
from typing import Iterator, Protocol, runtime_checkable
from graphforge.training.prompt import Message


@runtime_checkable
class Policy(Protocol):
    def sample(self, messages: list[Message]) -> str: ...


class ScriptedPolicy:
    def __init__(self, completions):
        self._iter = iter(completions)
    def sample(self, _messages):
        return next(self._iter)


class HfPolicy:
    def __init__(self, model, tokenizer, *, max_new_tokens=384, temperature=0.7, top_p=0.95):
        self.model = model
        self.tokenizer = tokenizer
        self.max_new_tokens = max_new_tokens
        self.temperature = temperature
        self.top_p = top_p

    def sample(self, messages):
        import torch
        tok = self.tokenizer
        text = tok.apply_chat_template(messages, add_generation_prompt=True, tokenize=False)
        inputs = tok(text, return_tensors="pt")
        inputs = {k: v.to(self.model.device) for k, v in inputs.items()}
        with torch.no_grad():
            out_ids = self.model.generate(
                **inputs,
                max_new_tokens=self.max_new_tokens,
                do_sample=True,
                temperature=self.temperature,
                top_p=self.top_p,
                pad_token_id=tok.eos_token_id,
            )
        prompt_len = inputs["input_ids"].shape[-1]
        gen = out_ids[0, prompt_len:]
        return tok.decode(gen, skip_special_tokens=True)
'''
pathlib.Path('graphforge/training/policy.py').write_text(_FIXED)

# Drop any cached graphforge imports so the next cell re-imports fresh.
_dropped = [k for k in list(sys.modules) if k.startswith('graphforge')]
for k in _dropped:
    del sys.modules[k]
print(f"Patched policy.py; cleared {len(_dropped)} cached imports.")

## 2. Run the full pipeline

`training.train.run` does baseline eval → trajectory generation → SFT → trained eval → plots, all in one call.

In [ ]:
from pathlib import Path
from training.config import TrainConfig
from training.train import run

cfg = TrainConfig(
    model_name='Qwen/Qwen2.5-0.5B-Instruct',
    task_id='t0.email_validator',
    n_oracle=20,
    n_explore=30,
    reward_threshold=5.0,
    epochs=2,
    learning_rate=1e-4,
    batch_size=1,
    gradient_accumulation_steps=4,
    use_lora=True,
    n_eval_episodes=20,
    out_dir=Path('outputs'),
    plots_dir=Path('plots'),
)
summary = run(cfg)
summary['baseline_eval']['mean_reward'], summary['trained_eval']['mean_reward']

## 3. Show the plots

In [ ]:
from IPython.display import Image, display
for name in ['comparison.png', 'baseline_rewards.png', 'trained_rewards.png', 'loss_curve.png']:
    p = Path('plots') / name
    if p.exists():
        print(name)
        display(Image(str(p)))

## 4. Commit the plots back to the repo

Once you're happy with the run, copy `plots/*.png` and `outputs/summary.json` into your fork and push. The README embeds them automatically.